# Session 7 — Your turn: Fine-tuning in practice

Check LoRA's two promises in numpy (it is small, and it can be merged), then write the per-field scoring used to judge an extraction model.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import json
import numpy as np

rng = np.random.default_rng(1)
d, r, alpha = 64, 4, 8
W = rng.normal(size=(d, d))                       # a frozen weight matrix: y = W x
A = rng.normal(size=(r, d)) * 0.1                 # LoRA: B A is a rank-r change
B = rng.normal(size=(d, r)) * 0.1
x = rng.normal(size=(5, d))
from aij_check import check, close, summary

## Task 1: LoRA: small, and mergeable

Compute `full_params` (the entries of `W`), `lora_params` (the entries of `A` and `B`) and `share` (`lora_params / full_params`). Then compute the layer's output **with the adapter on the side**,
`y_side = x @ W.T + (alpha / r) * (x @ A.T) @ B.T`, and **merged into the weights**, `W_merged = W + (alpha / r) * B @ A`, `y_merged = x @ W_merged.T`.

In [ ]:
full_params = ...
lora_params = ...
share = ...
y_side = ...
W_merged = ...
y_merged = ...

In [ ]:
check("the matrix has 4,096 entries", lambda: full_params == 4096)
check("the adapter has 512 entries, an eighth", lambda: lora_params == 512 and close(share, 0.125, 1e-12))
check("merging gives exactly the same output as the side path", lambda: close(y_side, y_merged, 1e-9))
check("the merged matrix has the shape of W, so serving costs nothing extra", lambda: np.shape(W_merged) == np.shape(W))
check("the change B A has rank at most r", lambda: np.linalg.matrix_rank(W_merged - W) <= r)
summary()

## Task 2: Score an extraction model field by field

A model was asked for JSON with the keys `robot`, `battery`, `charger`, `status`, `time`. Write `field_accuracy(outputs, truths)` where `outputs` is a list of strings (the model's answers, possibly not valid JSON) and `truths` a list of dictionaries.
It returns a dictionary with one accuracy per key plus `"all"` (the share of answers where every field is right). An answer that does not parse counts as wrong in every field.

In [ ]:
def field_accuracy(outputs, truths):
    ...

In [ ]:
truths = [{"robot": "R-1", "battery": 98, "charger": "C1", "status": "LOW", "time": "18:20"}, {"robot": "R-2", "battery": 40, "charger": "C2", "status": "OK", "time": "09:05"}, {"robot": "R-3", "battery": 7, "charger": "C3", "status": "LOW", "time": "23:59"}]
outputs = [json.dumps(truths[0]), json.dumps({**truths[1], "battery": "40%"}), "Sorry, I cannot do that."]
scores = field_accuracy(outputs, truths)
check("a field right in two answers of three scores 2/3, in one of three 1/3", lambda: close(scores["robot"], 2 / 3, 1e-9) and close(scores["battery"], 1 / 3, 1e-9))
check("an unparseable answer counts as wrong everywhere", lambda: all(scores[k] <= 2 / 3 for k in scores))
check("'all' counts only fully right answers", lambda: close(scores["all"], 1 / 3, 1e-9))
check("every key has an accuracy", lambda: set(scores) == {"robot", "battery", "charger", "status", "time", "all"})
summary()